# recs_004 task A (consumer mode) - _003

This notebook is analysis-only and consumes centralized Phase 1 eval artifacts.

## Prerequisites

Run pipeline first:
- `python scripts/recs_job_eval_query_embeddings.py configs/recs_job_eval_query_embeddings_phase1.json`

Optional baseline freeze:
- `python scripts/recs_job_eval_query_embeddings.py configs/recs_job_eval_query_embeddings_phase1.json --write-baseline`

Artifact source:
- `artifacts/recs/eval/`

In [ ]:
from pathlib import Path
import json

import pandas as pd


def _find_repo_root(start: Path) -> Path:
    here = start.resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from start={start}")


REPO_ROOT = _find_repo_root(Path.cwd())
EVAL_DIR = REPO_ROOT / "artifacts" / "recs" / "eval"

PATHS = {
    "overall": EVAL_DIR / "eval_phase1_overall.csv",
    "by_slice": EVAL_DIR / "eval_phase1_by_slice.csv",
    "by_support": EVAL_DIR / "eval_phase1_by_support_bucket.csv",
    "by_pop_decile": EVAL_DIR / "eval_phase1_by_pop_decile.csv",
    "pop_delta": EVAL_DIR / "eval_phase1_pop_delta_vs_popularity.csv",
    "personalization": EVAL_DIR / "eval_phase1_personalization.csv",
    "run_meta": EVAL_DIR / "eval_phase1_run_meta.json",
}

missing = [str(p) for p in PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing eval artifact(s):\n- " + "\n- ".join(missing))

tables = {k: pd.read_csv(v) for k, v in PATHS.items() if k != "run_meta"}
run_meta = json.loads(PATHS["run_meta"].read_text(encoding="utf-8"))

print("Loaded artifacts from", EVAL_DIR)
print("Methods:", run_meta.get("methods_run", []))
print("Examples:", run_meta.get("n_examples_evaluable"))

In [ ]:
print("## Overall leaderboard")
display(tables["overall"].sort_values(["NDCG@K", "MAP@K", "MRR"], ascending=False))

print("## Slice leaderboard")
display(tables["by_slice"].sort_values(["slice_name", "NDCG@K", "Hit@K"], ascending=[True, False, False]))

print("## Support-bucket leaderboard")
display(tables["by_support"].sort_values(["train_support_bucket", "NDCG@K"], ascending=[True, False]))

In [ ]:
print("## Popularity decile performance")
display(tables["by_pop_decile"].sort_values(["pos_pop_decile", "NDCG@K"], ascending=[True, False]))

print("## Delta vs popularity")
display(tables["pop_delta"].sort_values(["pos_pop_decile", "method"]))

print("## Personalization diagnostics")
display(tables["personalization"].sort_values("method"))

In [ ]:
print("## Run metadata")
run_meta